In [1]:
import glob, os, numpy as np
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker

# 1. Grab your files and build labels (same as before)
file_list = sorted(glob.glob(os.path.expanduser('~/Desktop/FINAL_brainmath_GLM_maps/*_GLM_zmap.nii.gz')))
mld_subs = ['059', '065', '067', '069', '071', '075', '076', '077', '078', '083', '088', '095', '096', '103', '106']
y = np.array([1 if any(f"sub-{sub}" in fname for sub in mld_subs) else 0 for fname in file_list])

# 2. Fetch the Schaefer 2018 atlas (Let's use 100 regions to start)
# You can change n_rois to 200, 300, 400, up to 1000.
schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7, resolution_mm=2)
atlas_filename = schaefer.maps

# 3. Initialize the Labels Masker using the Schaefer atlas
masker = NiftiLabelsMasker(labels_img=atlas_filename, standardize=True)

# 4. Extract features: This averages the voxels into exactly 100 regions per subject
X = masker.fit_transform(file_list)

print(f"Feature Matrix X shape: {X.shape} | Labels y shape: {y.shape}")

[fetch_atlas_schaefer_2018] Dataset found in /Users/jchong058/nilearn_data/schaefer_2018
[fetch_atlas_schaefer_2018] Downloading data from https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/brain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_100Parcels_7Networks_order.txt ...
[fetch_atlas_schaefer_2018]  ...done. (0 seconds, 0 min)

[fetch_atlas_schaefer_2018] Downloading data from https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/brain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_100Parcels_7Networks_order_FSLMNI152_2mm.nii.gz ...
[fetch_atlas_schaefer_2018]  ...done. (0 seconds, 0 min)



/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_44033/1722148345.py:19: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  X = masker.fit_transform(file_list)


Feature Matrix X shape: (239, 100) | Labels y shape: (239,)


In [2]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# 1. Setup Cross-Validation
# Stratified K-Fold ensures every fold has the same ratio of MLD to TD students
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Define the ML models to test
models = {
    "SVM (RBF Kernel)": SVC(kernel='rbf', random_state=42),
    "SVM (Linear)": SVC(kernel='linear', random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
}

# 3. Run the Gauntlet
print("--- 5-Fold Cross-Validation Accuracy ---")
for name, model in models.items():
    # The pipeline scales the data *inside* the CV loop to prevent data leakage
    pipeline = make_pipeline(StandardScaler(), model)
    
    # Calculate accuracy
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
    
    # Output the mean accuracy and standard deviation across the 5 folds
    print(f"{name:20s}: {scores.mean():.3f} (± {scores.std():.3f})")

--- 5-Fold Cross-Validation Accuracy ---
SVM (RBF Kernel)    : 0.607 (± 0.046)
SVM (Linear)        : 0.657 (± 0.028)
XGBoost             : 0.590 (± 0.065)
Random Forest       : 0.598 (± 0.056)
Logistic Regression : 0.666 (± 0.065)


In [3]:
# 1. Create true/false masks by checking the filenames
is_mult = np.array(['task-Mult' in f for f in file_list])
is_sub = np.array(['task-Sub' in f for f in file_list])

# 2. Slice the existing X and y matrices into two separate datasets
X_mult, y_mult = X[is_mult], y[is_mult]
X_sub, y_sub = X[is_sub], y[is_sub]

print(f"Multiplication Dataset: {X_mult.shape[0]} runs")
print(f"Subtraction Dataset: {X_sub.shape[0]} runs\n")

# 3. Define a quick function to run the models so we don't repeat code
def evaluate_task(X_task, y_task, task_name):
    print(f"--- 5-Fold CV Accuracy for {task_name} ---")
    for name, model in models.items():
        pipeline = make_pipeline(StandardScaler(), model)
        scores = cross_val_score(pipeline, X_task, y_task, cv=cv, scoring='accuracy')
        print(f"{name:20s}: {scores.mean():.3f} (± {scores.std():.3f})")
    print("\n")

# 4. Run the evaluation on both!
evaluate_task(X_mult, y_mult, "MULTIPLICATION")
evaluate_task(X_sub, y_sub, "SUBTRACTION")

Multiplication Dataset: 119 runs
Subtraction Dataset: 120 runs

--- 5-Fold CV Accuracy for MULTIPLICATION ---
SVM (RBF Kernel)    : 0.487 (± 0.031)
SVM (Linear)        : 0.547 (± 0.089)
XGBoost             : 0.480 (± 0.063)
Random Forest       : 0.512 (± 0.049)
Logistic Regression : 0.572 (± 0.046)


--- 5-Fold CV Accuracy for SUBTRACTION ---
SVM (RBF Kernel)    : 0.583 (± 0.065)
SVM (Linear)        : 0.633 (± 0.089)
XGBoost             : 0.650 (± 0.068)
Random Forest       : 0.583 (± 0.070)
Logistic Regression : 0.675 (± 0.061)


